In [43]:
from agents.voice import workflow
from langchain_openai import AzureChatOpenAI
from langgraph import graph
from langgraph.graph import START, END, StateGraph
import os
from dotenv import load_dotenv
from typing import TypedDict, final


In [44]:
load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT_NAME="gpt-4.1_1"

llm_model = AzureChatOpenAI(azure_endpoint=AZURE_OPENAI_ENDPOINT,
api_version=AZURE_OPENAI_API_VERSION, azure_deployment=AZURE_OPENAI_DEPLOYMENT_NAME, temperature=0.2,)

In [45]:
class BlogLLMChat(TypedDict):
    content: str
    title : str
    outline : str

In [49]:
def create_outline(state : BlogLLMChat):
    title = state["title"]
    prompt = f"Generate a detailed outline for a blog on the topic {title}"
    outline = llm_model.invoke(prompt).content
    state["outline"] = outline
    return state

def create_blog(state : BlogLLMChat) -> BlogLLMChat: #  or without use -> BlogLLMChat
    title = state["title"]
    outline = state["outline"]

    prompt = f'write a detailed outline blog on the title - {title} using following outline - {outline}'

    content = llm_model.invoke(prompt).content
    state["content"] = content
    return state

In [50]:
graph = StateGraph(BlogLLMChat)

graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)

graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", END)

workflow = graph.compile()

In [51]:
intial_state = {"title": "Rise of AI in india"}
final_state = workflow.invoke(intial_state)
final_state["content"]

'Certainly! Here’s a detailed outline for your blog, **"The Rise of AI in India: Transforming a Nation’s Future"**, based on your provided structure:\n\n---\n\n## I. Introduction\n\n- **Brief overview of Artificial Intelligence (AI):**\n  - Definition of AI and its core technologies (machine learning, deep learning, NLP, robotics).\n  - Examples of AI applications in everyday life.\n\n- **Global context: AI as a transformative technology:**\n  - AI’s impact on economies, industries, and societies worldwide.\n  - Comparison with previous technological revolutions (internet, mobile).\n\n- **India’s unique position and potential in the AI revolution:**\n  - Demographics: large population, young workforce.\n  - Rapid digital adoption and mobile penetration.\n  - Strategic importance for global AI development.\n\n- **Purpose and scope of the blog:**\n  - To explore India’s journey in AI, current landscape, sectoral impact, opportunities, challenges, and future prospects.\n\n---\n\n## II. Hi